In [ ]:
import matplotlib.pyplot as plt
import pandas
import functools 
import datetime
import numpy
import scipy

import pmana.utils
import pmana.purity

#### Look at the purity of one measurement

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

YEAR = '2026'
MONTH = 'Mar'
MEASUREMENT = f'08_15_48'
FILE_PATH = f"../../data/coldbox/{YEAR}_{MONTH}/Record_{YEAR}_{MONTH}_{MEASUREMENT}.csv"
# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = True,
    COL_NAMES = ['binCenter', 'F1', 'F2', 'F3', 'F4'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [1, 2, 3, 4]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
        rebin = True,
        debug = False,
        DISPLAY_FIT = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Pulse height [V]",
    "Counts [#]"
)

# ax.set_xlim(0.75, 0.9)
# ax.set_ylim(0, 500)

ax.legend()
# ax.set_yscale('log')
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT.replace('_','-')}")
plt.show()
fig.savefig(f"../../plots/coldbox/RawData_{YEAR}_{MONTH}.png", dpi=300, bbox_inches='tight')

#### Calibrate test pulses

In [ ]:
# configure analysis
ANALYSIS_CONFIGURATION = pmana.purity.config.DEFAULT_ANALYSIS_CONFIGURATION
ANALYSIS_CONFIGURATION['InnerLongChannel'] = 2
ANALYSIS_CONFIGURATION['OuterLongChannel'] = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 0
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 1

In [ ]:
# analyze campaign
TIME_DIR = f"2026_Mar"
PATH_CAMPAIGN = f"../../data/coldbox/{TIME_DIR}"

### test pulses
Output = pmana.utils.iterators.IterateCERN_CSV(
  PATH_CAMPAIGN,  ###< path to data
  functools.partial(
    pmana.utils.anatestdata.GaussianFitToChannel, 
    IS_DT5781=False, IS_CSV=True, rebin=False, debug=False,
    MASK_TESTPULSE = True, TESTPULSE_LOW_LIM = 0.7,
    DELIMITER = ',', SKIP_NROWS = 0, 
    BINNAME = 'BinCenter', COUNTNAME = 'Population'
  ),
  START_FROM = datetime.datetime(2026, 3, 7, 3, 43)
)

# convert top DataFrame
Output = pandas.DataFrame(Output)

# re-format the dataframe
Output.columns = ["Peak_CH1", "Err_Peak_CH1", "Width_CH1", "Err_Width_CH1",
                  "Peak_CH2", "Err_Peak_CH2", "Width_CH2", "Err_Width_CH2", 
                  "Peak_CH3", "Err_Peak_CH3", "Width_CH3", "Err_Width_CH3", 
                  "Peak_CH4", "Err_Peak_CH4", "Width_CH4", "Err_Width_CH4", 
                  "Date"]

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2*2), nrows=2, layout='tight')

ax[0].plot(Output.Date, Output.Peak_CH1, lw=1, c='C0')
ax[0].fill_between(Output.Date, Output.Peak_CH1-Output.Err_Peak_CH1,  Output.Peak_CH1+Output.Err_Peak_CH1, alpha=0.3, fc='C0', label='inner-short')
ax[0].plot(Output.Date, Output.Peak_CH2, lw=1, c='C1')
ax[0].fill_between(Output.Date, Output.Peak_CH2-Output.Err_Peak_CH2,  Output.Peak_CH2+Output.Err_Peak_CH2, alpha=0.3, fc='C1', label='outer-short')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[0], None, 'Test-pulse [V]')
ax[0].set_xticklabels([])

ax[1].plot(Output.Date, Output.Peak_CH3, lw=1, c='C0')
ax[1].fill_between(Output.Date, Output.Peak_CH3-Output.Err_Peak_CH3,  Output.Peak_CH3+Output.Err_Peak_CH3, alpha=0.3, fc='C0', label='inner-long')
ax[1].plot(Output.Date, Output.Peak_CH4, lw=1, c='C1')
ax[1].fill_between(Output.Date, Output.Peak_CH4-Output.Err_Peak_CH4,  Output.Peak_CH4+Output.Err_Peak_CH4, alpha=0.3, fc='C1', label='outer-long')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[1], None, 'Test-pulse [V]')

# ax.set_yscale('log')
# ax.set_ylim(170, 240)

# gfx
for a in ax:
  a.legend(handlelength=1, loc='upper right', fancybox=False)
  a.tick_params(axis='x', labelrotation=30)
  a.grid(which='major', alpha=0.5)
  a.grid(which='minor', ls='--', alpha=0.3)

fig.suptitle(f'CERN-Coldbox data', fontsize=16)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), nrows=1, layout='tight')

# short PrM
Output['InnerOverOuter_Short'] = Output['Peak_CH1'] / Output['Peak_CH2']
ax.plot(Output['Date'], Output['InnerOverOuter_Short'], c='C0', label='Short Bi-PrM')
MedianRatio = numpy.median(Output[Output['Date'] > datetime.datetime(2026, 3, 7, 20)]['InnerOverOuter_Short'])
ax.axhline(MedianRatio, c='C0', ls=':', label=f'median: {MedianRatio:.3f}')
print('Short Bi-PrM inner-over-outer test-pulse ratio:', MedianRatio)

# long PrM
Output['InnerOverOuter_Long'] = Output['Peak_CH3'] / Output['Peak_CH4']
ax.plot(Output['Date'], Output['InnerOverOuter_Long'], c='C1', label='Long Bi-PrM')
MedianRatio = numpy.median(Output[Output['Date'] > datetime.datetime(2026, 3, 7, 20)]['InnerOverOuter_Long'])
ax.axhline(MedianRatio, c='C1', ls=':', label=f'median: {MedianRatio:.3f}')
print('Long Bi-PrM inner-over-outer test-pulse ratio:', MedianRatio)

ax.set_xlim(Output['Date'].min(), Output['Date'].max())
# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Inner/Outer')
ax.legend(handlelength=1, loc='center left', fancybox=False)
ax.tick_params(axis='x', labelrotation=30)
ax.grid(which='major', alpha=0.5)
ax.grid(which='minor', ls='--', alpha=0.3)

fig.suptitle(f'CERN-Coldbox data / test-pulse', fontsize=16)

plt.show()
fig.savefig(f"../../plots/coldbox/TestPulseRatio_{TIME_DIR}.png", dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), nrows=1, layout='tight')

# short PrM
Output['InnerOverInner'] = Output['Peak_CH1'] / Output['Peak_CH3']
ax.plot(Output['Date'], Output['InnerOverInner'], c='C0', label='inner anode')
# MedianRatio = numpy.median(Output[Output['Date'] > datetime.datetime(2026, 3, 7, 20)]['InnerOverOuter_Short'])
# ax.axhline(MedianRatio, c='C0', ls=':', label=f'median: {MedianRatio:.3f}')
# print('Short Bi-PrM inner-over-outer test-pulse ratio:', MedianRatio)

# long PrM
Output['OuterOverOuter'] = Output['Peak_CH2'] / Output['Peak_CH4']
ax.plot(Output['Date'], Output['OuterOverOuter'], c='C1', label='outer anode')
# MedianRatio = numpy.median(Output[Output['Date'] > datetime.datetime(2026, 3, 7, 20)]['InnerOverOuter_Long'])
# ax.axhline(MedianRatio, c='C1', ls=':', label=f'median: {MedianRatio:.3f}')
# print('Long Bi-PrM inner-over-outer test-pulse ratio:', MedianRatio)

ax.set_xlim(Output['Date'].min(), Output['Date'].max())
# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Short/Long')
ax.legend(handlelength=1, loc='center left', fancybox=False)
ax.tick_params(axis='x', labelrotation=30)
ax.grid(which='major', alpha=0.5)
ax.grid(which='minor', ls='--', alpha=0.3)

fig.suptitle(f'CERN-Coldbox data / test-pulse', fontsize=16)

plt.show()
fig.savefig(f"../../plots/coldbox/TestPulseRatio_2_{TIME_DIR}.png", dpi=300, bbox_inches='tight')

In [ ]:
# configure calibration factors
# the important thing is inter-calibrating
# inner and outer anodes within a PrM
# e.g., by aligning their test-pulse
ANALYSIS_CONFIGURATION['InnerLongCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterLongCalibration'] = 1.04389
ANALYSIS_CONFIGURATION['InnerShortCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterShortCalibration'] = 1.02582

#### Purity analysis

In [ ]:
# configure analysis
ANALYSIS_CONFIGURATION = pmana.purity.config.DEFAULT_ANALYSIS_CONFIGURATION
ANALYSIS_CONFIGURATION['LongICPeakSearchLimits'] = (0.15, 0.5)   # long Bi-PrM has a significant tail...
ANALYSIS_CONFIGURATION['ShortICPeakSearchLimits'] = (0.4, 0.8)  # short Bi-PrM has a test pulse...
ANALYSIS_CONFIGURATION['InnerLongChannel'] = 2
ANALYSIS_CONFIGURATION['OuterLongChannel'] = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 0
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 1

# configure calibration factors
# the important thing is inter-calibrating
# inner and outer anodes within a PrM
# e.g., by aligning their test-pulse
ANALYSIS_CONFIGURATION['InnerLongCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterLongCalibration'] = 1.
ANALYSIS_CONFIGURATION['InnerShortCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterShortCalibration'] = 1. #1.02582

In [ ]:
# analyze campaign
TIME_DIR = f"2026_Mar"
PATH_CAMPAIGN = f"../../data/coldbox/{TIME_DIR}"

### long PrM
Output = pmana.utils.iterators.IterateCERN_CSV(
    PATH_CAMPAIGN,  ###< path to data
    # pmana.utils.anatestdata.AnalyzeMeasurement, 
    functools.partial(
        pmana.purity.ana.ExtractICPeak, 
        PM_TAG = 'Long',
        ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
    ),  ###< analyzing module, changing some options 
)

# convert top DataFrame and re-format the dataframe
Output = pandas.DataFrame(Output)
Output.columns = [
  "LongIC_Peak", "LongIC_PeakErr", "LongIC_Scale",
  "Date"
]

### short PrM
Output_Short = pmana.utils.iterators.IterateCERN_CSV(
  PATH_CAMPAIGN,  ###< path to data
  # pmana.utils.anatestdata.AnalyzeMeasurement, 
  functools.partial(
      pmana.purity.ana.ExtractICPeak, 
      PM_TAG = 'Short',
      ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
  ),  ###< analyzing module, changing some options 
)

# convert top DataFrame and re-format the dataframe
Output_Short = pandas.DataFrame(Output_Short)
Output_Short.columns = [
  "ShortIC_Peak", "ShortIC_PeakErr", "ShortIC_Scale",
  "Date"
]

### merge
Output = pandas.merge(Output, Output_Short, on = 'Date', how='inner')

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
Output['Lifetime'], Output['Lifetime_Err'] = pmana.purity.ana.GetLifetime_DoublePrM(
  Output.ShortIC_Peak, Output.LongIC_Peak,
  Output.ShortIC_PeakErr, Output.LongIC_PeakErr,
  SHORT_DRIFT_LENGTH = 40, LONG_DRIFT_LENGTH = 400
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), nrows=1, layout='tight')

ax.errorbar(
  Output['Date'], Output['Lifetime'], 
  yerr = Output['Lifetime_Err'],
  c='black', label='Double Bi-PrM (4 + 40 cm)',
  ls='', marker='.'
)

# ax.set_yscale('log')
ax.set_ylim(170, 240)
ax.set_xlim(Output['Date'].min(), Output['Date'].max())

# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Lifetime [μs]')
ax.legend(handlelength=1, loc='upper right', fancybox=False)
ax.tick_params(axis='x', labelrotation=30)
ax.grid(which='major', alpha=0.5)
ax.grid(which='minor', ls='--', alpha=0.3)

fig.suptitle(f'CERN-Coldbox data', fontsize=16)

plt.show()
fig.savefig(f"../../plots/coldbox/Lifetime_{TIME_DIR}.png", dpi=300, bbox_inches='tight')

##### Purity analysis, but save spectra

In [ ]:
# configure analysis
ANALYSIS_CONFIGURATION = pmana.purity.config.DEFAULT_ANALYSIS_CONFIGURATION
ANALYSIS_CONFIGURATION['LongICPeakSearchLimits'] = (0.15, 0.5)   # long Bi-PrM has a significant tail...
ANALYSIS_CONFIGURATION['ShortICPeakSearchLimits'] = (0.4, 0.8)  # short Bi-PrM has a test pulse...
ANALYSIS_CONFIGURATION['InnerLongChannel'] = 2
ANALYSIS_CONFIGURATION['OuterLongChannel'] = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 0
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 1

# configure calibration factors
# the important thing is inter-calibrating
# inner and outer anodes within a PrM
# e.g., by aligning their test-pulse
ANALYSIS_CONFIGURATION['InnerLongCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterLongCalibration'] = 1.04389
ANALYSIS_CONFIGURATION['InnerShortCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterShortCalibration'] = 1.02582

In [ ]:
# analyze campaign
TIME_DIR = f"2026_Mar"
PATH_CAMPAIGN = f"../../data/coldbox/{TIME_DIR}"

Output = pmana.utils.iterators.IterateCERN_CSV(
    PATH_CAMPAIGN,  ###< path to data
    # pmana.utils.anatestdata.AnalyzeMeasurement, 
    functools.partial(
        pmana.purity.ana.ExtractICPeak, 
        PM_TAG = 'Long',
        SAVE_SPECTRA = True, ###< save the spectra on the dataframe
        ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
    )  ###< analyzing module, changing some options 
)

# convert top DataFrame
Output = pandas.DataFrame(Output)

# re-format the dataframe
Output.columns = [
  "LongIC_Peak", "LongIC_PeakErr", "LongIC_Scale",
  "xIC", "IC",
  "Date"
]

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

YEAR = '2026'
MONTH = 'Mar'
MEASUREMENT = f'08_15_48'
FILE_PATH = f"../../data/coldbox/{YEAR}_{MONTH}/Record_{YEAR}_{MONTH}_{MEASUREMENT}.csv"

# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = True,
    COL_NAMES = ['binCenter', 'F1', 'F2', 'F3', 'F4'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [
    ANALYSIS_CONFIGURATION['InnerLongChannel']+1,
    ANALYSIS_CONFIGURATION['OuterLongChannel']+1 
]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
        rebin = False,
        debug = False,
        DISPLAY_FIT = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Integral [V*t/R] / Pulse height [V]",
    "Counts [#]"
)

# plot the corresponding IC peak
MONTH_ABBR = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
DAY, HOUR, MINUTE = MEASUREMENT.split('_')
EXACT_DATE = datetime.datetime(
    int(YEAR), 
    int(MONTH_ABBR[MONTH]),
    int(DAY),
    int(HOUR),
    int(MINUTE)
)
Mask = Output['Date'] == EXACT_DATE
print("Results for this measurement:\n", Output[Mask])
xIC = Output[Mask].iloc[0]['xIC']
IC = Output[Mask].iloc[0]['IC']
ax.step(
    xIC, 
    IC,
    where = 'mid',
    c = 'red',
    lw = 1,
    label = 'IC'
)

ax.legend(handlelength=1, loc='upper right', frameon=True, fancybox=False)
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT}")
plt.show()
fig.savefig(f"../../plots/coldbox/RawData_IC_{YEAR}_{MONTH}.png", dpi=300, bbox_inches='tight')

In [ ]:
Output = pmana.utils.iterators.IterateCERN_CSV(
  PATH_CAMPAIGN,  ###< path to data
  # pmana.utils.anatestdata.AnalyzeMeasurement, 
  functools.partial(
      pmana.purity.ana.ExtractICPeak, 
      PM_TAG = 'Short',
      SAVE_SPECTRA = True, ###< save the spectra on the dataframe
      ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
  ),  ###< analyzing module, changing some options 
)

# convert top DataFrame
Output = pandas.DataFrame(Output)

# re-format the dataframe
Output.columns = [
  "ShortIC_Peak", "ShortIC_PeakErr", "ShortIC_Scale",
  "xIC", "IC",
  "Date"
]

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = True,
    COL_NAMES = ['binCenter', 'F1', 'F2', 'F3', 'F4'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [
    ANALYSIS_CONFIGURATION['InnerShortChannel']+1,
    ANALYSIS_CONFIGURATION['OuterShortChannel']+1 
]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
        rebin = False,
        debug = False,
        DISPLAY_FIT = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Pulse height [V]",
    "Counts [#]"
)

# plot the corresponding IC peak
MONTH_ABBR = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
DAY, HOUR, MINUTE = MEASUREMENT.split('_')
EXACT_DATE = datetime.datetime(
    int(YEAR), 
    int(MONTH_ABBR[MONTH]),
    int(DAY),
    int(HOUR),
    int(MINUTE)
)
Mask = Output['Date'] == EXACT_DATE
print("Results for this measurement:\n", Output[Mask])
xIC = Output[Mask].iloc[0]['xIC']
IC = Output[Mask].iloc[0]['IC']
ax.step(
    xIC, 
    IC,
    where = 'mid',
    c = 'red',
    lw = 1,
    label = 'IC'
)

ax.legend(handlelength=1, loc='upper right', frameon=True, fancybox=False)
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT}")
plt.show()
fig.savefig(f"../../plots/cern/RawData_IC_Short_{YEAR}_{MONTH}.png", dpi=300, bbox_inches='tight')

#### Single Bi-PrM operation

In [ ]:
# we consider the lifetime measured with the double Bi-PrM setup
# the "true" lifetime, and use it as reference to be able to extract
# the asymptotic voltage for the short Bi-PrM
# this is especially useful as the lifetime is decreasing (due to no argon purification)
# and eventually the IC peak won't be as clear in the long Bi-PrM
# (vice versa, a similar thing happens for the short Bi-PrM for very high lifetimes)

In [ ]:
DRIFT_TIME = 40 / 1.547
x    = numpy.exp(- DRIFT_TIME / Output['Lifetime'])
y    = Output['ShortIC_Peak']
yerr = Output['ShortIC_PeakErr']

fig, ax = plt.subplots(figsize=(4, 3), nrows=1, layout='constrained')

# measurement
ax.errorbar(
  x, y, 
  yerr = yerr,
  c='black', label='Short Bi-PrM (4 cm)',
  ls='', marker='.'
)

# fit
def line(x, a):
  return a * x 

pars, covs = scipy.optimize.curve_fit(
  line,
  xdata = x,
  ydata = y,
  sigma = yerr,
  absolute_sigma = True
)
errs = numpy.sqrt(numpy.diag(covs))
xs = numpy.linspace(min(x), max(x), 100)
ax.plot(xs, pars[0]*xs, c='red', lw=2, label='$V_{asym.}$='+f'{pars[0]:.5f} V')

# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, 'exp(-t/τ)', 'IC peak [V]')
ax.legend(handlelength=1, loc='upper left', fancybox=False)

ax.set_title(f'CERN-Coldbox data', fontsize=16)
fig.savefig(f"../../plots/coldbox/SinglePrM_AsymptoticIC_{TIME_DIR}.png", dpi=300, bbox_inches='tight')

In [ ]:
ANALYSIS_CONFIGURATION['ShortAsymptoticICPeak'] = pars[0]

Output['Lifetime_Short'] = pmana.purity.ana.GetLifetime_SinglePrM(
  Output['ShortIC_Peak'],
  ANALYSIS_CONFIGURATION['ShortAsymptoticICPeak'],
  DRIFT_LENGTH = 40
)

In [ ]:
DRIFT_TIME = 400 / 1.547
x    = numpy.exp(- DRIFT_TIME / Output['Lifetime'])
y    = Output['LongIC_Peak']
yerr = Output['LongIC_PeakErr']

fig, ax = plt.subplots(figsize=(4, 3), nrows=1, layout='constrained')

# measurement
ax.errorbar(
  x, y, 
  yerr = yerr,
  c='black', label='Long Bi-PrM (40 cm)',
  ls='', marker='.'
)

# fit
def line(x, a):
  return a * x 

pars, covs = scipy.optimize.curve_fit(
  line,
  xdata = x,
  ydata = y,
  sigma = yerr,
  absolute_sigma = True
)
errs = numpy.sqrt(numpy.diag(covs))
xs = numpy.linspace(min(x), max(x), 100)
ax.plot(xs, pars[0]*xs, c='red', lw=2, label='$V_{asym.}$='+f'{pars[0]:.5f} V')

# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, 'exp(-t/τ)', 'IC peak [V]')
ax.legend(handlelength=1, loc='upper left', fancybox=False)

ax.set_title(f'CERN-Coldbox data', fontsize=16)
fig.savefig(f"../../plots/coldbox/LongPrM_AsymptoticIC_{TIME_DIR}.png", dpi=300, bbox_inches='tight')

In [ ]:
ANALYSIS_CONFIGURATION['LongAsymptoticICPeak'] = pars[0]

Output['Lifetime_Long'] = pmana.purity.ana.GetLifetime_SinglePrM(
  Output['LongIC_Peak'],
  ANALYSIS_CONFIGURATION['LongAsymptoticICPeak'],
  DRIFT_LENGTH = 400
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), nrows=1, layout='tight')

# double PrM
ax.errorbar(
  Output['Date'], Output['Lifetime'], 
  yerr = Output['Lifetime_Err'],
  c='black', label='Double Bi-PrM (4 + 40 cm)',
  ls='', marker='.'
)

# single PrM operation
ax.plot(
  Output['Date'], Output['Lifetime_Short'], 
  c='C0', label='Short Bi-PrM (4 cm)\n$V_{asym.}$='+f'{ANALYSIS_CONFIGURATION['ShortAsymptoticICPeak']:.5f} V',
  ls='-', lw=2
)

ax.plot(
  Output['Date'], Output['Lifetime_Long'], 
  c='C1', label='Long Bi-PrM (40 cm)\n$V_{asym.}$='+f'{ANALYSIS_CONFIGURATION['LongAsymptoticICPeak']:.5f} V',
  ls='-', lw=2
)

# ax.set_yscale('log')
ax.set_ylim(120, 240)
ax.set_xlim(Output['Date'].min(), Output['Date'].max())

# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Lifetime [μs]')
ax.legend(handlelength=1, loc='lower left', fancybox=False)
ax.tick_params(axis='x', labelrotation=30)
ax.grid(which='major', alpha=0.5)
ax.grid(which='minor', ls='--', alpha=0.3)

fig.suptitle(f'CERN-Coldbox data', fontsize=16)

plt.show()
fig.savefig(f"../../plots/coldbox/Lifetime_WSingle_{TIME_DIR}.png", dpi=300, bbox_inches='tight')